**Build Individual-Gene Loss Cohorts**
This notebook prepares the next phase of the project: individual-gene and small combined-gene analyses.

Instead of querying GDC again, split histone modifier and SWI/SNF pancancer event files into smaller cohorts:
- Single histone modifier genes: KDM6A, KDM6B, NSD1, NSD2, SETD2
- Combined histone models:      KDM6A/B and NSD1/2
- Single SWI/SNF genes:         ARID1A, ARID1B, SMARCA4, SMARCB1, PBRM1
- Combined SWI/SNF model:       ARID1A/B

Much faster than rerunning mutation and copy-number calling for every individual gene, because the existing group-level files already contain one row per LoF/HomDel event with the altered gene recorded.

In [ ]:
# Use same personal R library used by the TCGA notebooks

user_lib <- "/home/kennes38/ResearchProject/R/x86_64-pc-linux-gnu-library/4.5"
.libPaths(c(user_lib, .libPaths()))

library(dplyr)
library(readr)
library(tibble)
library(stringr)
library(purrr)

project_root <- "/home/kennes38/ResearchProject"
verification_dir <- file.path(project_root, "GENCODE_coordinate_verification")
dir.create(verification_dir, showWarnings = FALSE, recursive = TRUE)
setwd(verification_dir)

getwd()

In [ ]:
original_coordinates <- tibble::tribble(
  ~gene_set,           ~gene,     ~old_chr, ~old_start, ~old_end,
  "PRC2_core",         "EZH2",    "7",      148504460, 148525000,
  "PRC2_core",         "EED",     "11",       8590000,   8620000,
  "PRC2_core",         "SUZ12",   "17",      30240000,  30290000,
  "PRC2_core",         "RBBP4",   "1",       32600000,  32650000,
  "PRC2_core",         "RBBP7",   "X",       16700000,  16750000,
  "histone_modifier",  "KDM6A",   "X",       44873188,  45112779,
  "histone_modifier",  "KDM6B",   "17",       7834217,   7854796,
  "histone_modifier",  "NSD1",    "5",      177131787, 177300213,
  "histone_modifier",  "NSD2",    "4",        1871393,   1982192,
  "histone_modifier",  "SETD2",   "3",       47016436,  47164840,
  "SWI_SNF",           "ARID1A",  "1",       26696015,  26782104,
  "SWI_SNF",           "ARID1B",  "6",      156776026, 157210779,
  "SWI_SNF",           "SMARCA4", "19",      10960932,  11079426,
  "SWI_SNF",           "SMARCB1", "22",      23786966,  23838009,
  "SWI_SNF",           "PBRM1",   "3",       52545367,  52685913
)

expected_gene_identity <- tibble::tribble(
  ~gene,     ~expected_ensembl_gene_id, ~expected_chr,
  "EZH2",    "ENSG00000106462",         "7",
  "EED",     "ENSG00000074266",         "11",
  "SUZ12",   "ENSG00000178691",         "17",
  "RBBP4",   "ENSG00000162521",         "1",
  "RBBP7",   "ENSG00000102054",         "X",
  "KDM6A",   "ENSG00000147050",         "X",
  "KDM6B",   "ENSG00000132510",         "17",
  "NSD1",    "ENSG00000165671",         "5",
  "NSD2",    "ENSG00000109685",         "4",
  "SETD2",   "ENSG00000181555",         "3",
  "ARID1A",  "ENSG00000117713",         "1",
  "ARID1B",  "ENSG00000049618",         "6",
  "SMARCA4", "ENSG00000127616",         "19",
  "SMARCB1", "ENSG00000099956",         "22",
  "PBRM1",   "ENSG00000163939",         "3"
)

target_genes <- original_coordinates$gene

stopifnot(length(target_genes) == 15)
stopifnot(n_distinct(target_genes) == 15)
stopifnot(setequal(target_genes, expected_gene_identity$gene))

original_coordinates %>%
  left_join(expected_gene_identity, by = "gene")

In [ ]:
# GENCODE Human Release 46 is based on GRCh38.p14 and corresponds to Ensembl 112

gencode_release <- 46
gencode_assembly <- "GRCh38.p14"
gencode_url <- paste0(
  "https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_",
  gencode_release,
  "/gencode.v", gencode_release, ".annotation.gtf.gz"
)

gencode_file <- paste0("gencode.v", gencode_release, ".annotation.gtf.gz")

if (!file.exists(gencode_file)) {
  message("Downloading GENCODE v", gencode_release, " annotation...")
  download.file(gencode_url, destfile = gencode_file, mode = "wb")
} else {
  message("GENCODE file already present: ", gencode_file)
}

stopifnot(file.exists(gencode_file))

# Record file size and MD5 checksum so the exact downloaded file is auditable
gencode_file_record <- tibble(
  release = gencode_release,
  assembly = gencode_assembly,
  source_url = gencode_url,
  local_file = normalizePath(gencode_file),
  file_size_mb = file.info(gencode_file)$size / 1024^2,
  md5 = unname(tools::md5sum(gencode_file)),
  accessed_on = as.character(Sys.Date())
)

write_csv(gencode_file_record, "GENCODE_v46_download_record.csv")
gencode_file_record

In [ ]:
# GTF file has nine tab-separated columns
# Retain gene-level records -> then extract  gene symbol, stable Ensembl gene ID and gene type

gtf_columns <- c(
  "chromosome", "source", "feature_type", "start", "end",
  "score", "strand", "frame", "attributes"
)

gencode_gene_records <- read_tsv(
  gencode_file,
  comment = "#",
  col_names = gtf_columns,
  col_types = cols(
    chromosome = col_character(),
    source = col_character(),
    feature_type = col_character(),
    start = col_double(),
    end = col_double(),
    score = col_character(),
    strand = col_character(),
    frame = col_character(),
    attributes = col_character()
  ),
  progress = FALSE
) %>%
  filter(feature_type == "gene") %>%
  mutate(
    gene_id_versioned = str_match(attributes, 'gene_id "([^"]+)"')[, 2],
    gene_id = str_remove(gene_id_versioned, "\\.[0-9]+$"),
    gene_name = str_match(attributes, 'gene_name "([^"]+)"')[, 2],
    gene_type = str_match(attributes, 'gene_type "([^"]+)"')[, 2],
    chromosome = str_remove(chromosome, "^chr")
  ) %>%
  select(
    gene_name, gene_id, gene_id_versioned, gene_type,
    chromosome, start, end, strand, source
  )

cat("GENCODE gene records read:", nrow(gencode_gene_records), "\n")
head(gencode_gene_records)

In [ ]:
gencode_target_coordinates <- gencode_gene_records %>%
  filter(gene_name %in% target_genes) %>%
  left_join(expected_gene_identity, by = c("gene_name" = "gene")) %>%
  mutate(
    gene_id_matches_expected = gene_id == expected_ensembl_gene_id,
    chromosome_matches_expected = chromosome == expected_chr,
    interval_is_valid = !is.na(start) & !is.na(end) & start >= 1 & end >= start,
    record_is_protein_coding = gene_type == "protein_coding",
    validation_pass =
      gene_id_matches_expected &
      chromosome_matches_expected &
      interval_is_valid &
      record_is_protein_coding
  ) %>%
  arrange(match(gene_name, target_genes))

# Validation checks
missing_genes <- setdiff(target_genes, gencode_target_coordinates$gene_name)
duplicated_genes <- gencode_target_coordinates %>% count(gene_name) %>% filter(n != 1)
failed_identity_checks <- gencode_target_coordinates %>% filter(!validation_pass)

cat("Target genes requested:", length(target_genes), "\n")
cat("Target gene records found:", nrow(gencode_target_coordinates), "\n")
cat("Missing genes:", length(missing_genes), "\n")
cat("Genes with duplicate/non-single records:", nrow(duplicated_genes), "\n")
cat("Records failing identity/coordinate validation:", nrow(failed_identity_checks), "\n")

if (length(missing_genes) > 0) print(missing_genes)
if (nrow(duplicated_genes) > 0) print(duplicated_genes)
if (nrow(failed_identity_checks) > 0) {
  print(
    failed_identity_checks %>%
      select(
        gene_name, gene_id, expected_ensembl_gene_id,
        chromosome, expected_chr, gene_type, start, end,
        gene_id_matches_expected, chromosome_matches_expected,
        interval_is_valid, record_is_protein_coding
      )
  )
}

stopifnot(length(missing_genes) == 0)
stopifnot(nrow(gencode_target_coordinates) == 15)
stopifnot(nrow(duplicated_genes) == 0)
stopifnot(nrow(failed_identity_checks) == 0)
stopifnot(all(gencode_target_coordinates$validation_pass))

# This is the main verification table
gencode_target_coordinates %>%
  select(
    gene_name, gene_id, expected_ensembl_gene_id,
    gene_type, chromosome, expected_chr,
    start, end, strand,
    gene_id_matches_expected, chromosome_matches_expected,
    interval_is_valid, record_is_protein_coding, validation_pass
  )

In [ ]:
coordinate_comparison <- original_coordinates %>%
  left_join(
    gencode_target_coordinates %>%
      transmute(
        gene = gene_name,
        gencode_gene_id = gene_id,
        gencode_gene_id_versioned = gene_id_versioned,
        gencode_gene_type = gene_type,
        gencode_chr = chromosome,
        gencode_start = start,
        gencode_end = end,
        gencode_strand = strand
      ),
    by = "gene"
  ) %>%
  mutate(
    same_chromosome = old_chr == gencode_chr,
    intervals_overlap =
      same_chromosome & old_start <= gencode_end & old_end >= gencode_start,
    exact_match =
      same_chromosome & old_start == gencode_start & old_end == gencode_end,
    old_start_minus_gencode = old_start - gencode_start,
    old_end_minus_gencode = old_end - gencode_end,
    overlap_start = pmax(old_start, gencode_start),
    overlap_end = pmin(old_end, gencode_end),
    overlap_bp = if_else(intervals_overlap, overlap_end - overlap_start + 1, 0),
    gencode_gene_length_bp = gencode_end - gencode_start + 1,
    proportion_gencode_gene_covered_by_old_interval = overlap_bp / gencode_gene_length_bp,
    coordinate_status = case_when(
      exact_match ~ "exact_match",
      intervals_overlap ~ "partial_overlap",
      same_chromosome ~ "same_chromosome_no_overlap",
      TRUE ~ "different_chromosome"
    )
  )

write_csv(coordinate_comparison, "original_vs_GENCODE_v46_coordinate_comparison.csv")

coordinate_comparison %>%
  select(
    gene_set, gene,
    old_chr, old_start, old_end,
    gencode_gene_id, gencode_chr, gencode_start, gencode_end,
    coordinate_status, proportion_gencode_gene_covered_by_old_interval,
    old_start_minus_gencode, old_end_minus_gencode
  )

In [ ]:
# Final coordinate table is accepted only when all 15 records pass checks
stopifnot(nrow(gencode_target_coordinates) == 15)
stopifnot(all(gencode_target_coordinates$validation_pass))

final_verified_coordinates <- original_coordinates %>%
  select(gene_set, gene) %>%
  left_join(
    gencode_target_coordinates %>%
      transmute(
        gene = gene_name,
        ensembl_gene_id = gene_id,
        chromosome = chromosome,
        start = as.integer(start),
        end = as.integer(end),
        strand = strand,
        gene_type = gene_type,
        validation_pass = validation_pass
      ),
    by = "gene"
  ) %>%
  mutate(
    annotation_source = "GENCODE Human Release 46 comprehensive gene annotation",
    genome_assembly = "GRCh38.p14",
    verification_status = "GENCODE_v46_identity_and_interval_checks_passed"
  )

stopifnot(nrow(final_verified_coordinates) == 15)
stopifnot(!anyNA(final_verified_coordinates[, c("gene", "ensembl_gene_id", "chromosome", "start", "end")]))
stopifnot(all(final_verified_coordinates$validation_pass))

write_csv(
  final_verified_coordinates,
  "GENCODE_v46_GRCh38_verified_coordinates_all_15_genes.csv"
)

final_verified_coordinates

In [ ]:
# Output formatted to replace the manually typed coordinate tribbles in the TCGA loss-calling notebooks

pipeline_coordinates <- final_verified_coordinates %>%
  transmute(
    gene_set,
    gene,
    chr = chromosome,
    start,
    end
  )

write_csv(
  pipeline_coordinates,
  "CNV_pipeline_coordinates_GENCODE_v46_GRCh38.csv"
)

# Print copy-ready R code for each gene set
print_coordinate_tribble <- function(df, object_name = "target_coords") {
  cat(object_name, " <- tibble::tribble(\n", sep = "")
  cat("  ~gene, ~chr, ~start, ~end,\n")

  for (i in seq_len(nrow(df))) {
    comma <- if (i < nrow(df)) "," else ""
    cat(sprintf(
      '  "%s", "%s", %d, %d%s\n',
      df$gene[i], df$chr[i], df$start[i], df$end[i], comma
    ))
  }

  cat(")\n")
}

for (set_name in unique(pipeline_coordinates$gene_set)) {
  cat("\n# ", set_name, "\n", sep = "")
  print_coordinate_tribble(
    pipeline_coordinates %>% filter(gene_set == set_name),
    object_name = "target_coords"
  )
}

In [ ]:
audit_summary <- tibble(
  check = c(
    "15 unique target genes requested",
    "15 unique GENCODE gene records found",
    "All stable Ensembl gene IDs match expected IDs",
    "All chromosomes match expected chromosomes",
    "All target records are protein-coding",
    "All genomic intervals are valid",
    "All 15 combined validation checks passed",
    "GENCODE source file checksum recorded",
    "Final coordinate CSV written"
  ),
  passed = c(
    length(target_genes) == 15 && n_distinct(target_genes) == 15,
    nrow(gencode_target_coordinates) == 15 && nrow(duplicated_genes) == 0,
    all(gencode_target_coordinates$gene_id_matches_expected),
    all(gencode_target_coordinates$chromosome_matches_expected),
    all(gencode_target_coordinates$record_is_protein_coding),
    all(gencode_target_coordinates$interval_is_valid),
    all(gencode_target_coordinates$validation_pass),
    file.exists("GENCODE_v46_download_record.csv") && !is.na(gencode_file_record$md5),
    file.exists("GENCODE_v46_GRCh38_verified_coordinates_all_15_genes.csv")
  )
)

write_csv(audit_summary, "GENCODE_coordinate_verification_audit_summary.csv")
audit_summary

if (!all(audit_summary$passed)) {
  warning("One or more GENCODE validation checks failed. Do not use the coordinate table until investigated.")
} else {
  message("All GENCODE coordinate validation checks passed for all 15 genes.")
}

cat("\nCreated files:\n")
list.files(pattern = "\\.csv$", full.names = FALSE)

In [ ]:
coordinate_comparison %>%
  select(
    gene_set,
    gene,
    old_chr,
    old_start,
    old_end,
    gencode_chr,
    gencode_start,
    gencode_end,
    coordinate_status,
    proportion_gencode_gene_covered_by_old_interval,
    exact_match
  ) %>%
  arrange(gene_set, gene)

In [ ]:
coordinate_comparison %>%
  count(coordinate_status)

In [ ]:
coordinate_comparison %>%
  summarise(
    n_genes = n(),
    n_exact_matches = sum(exact_match),
    n_partial_overlaps = sum(coordinate_status == "partial_overlap"),
    n_same_chr_no_overlap =
      sum(coordinate_status == "same_chromosome_no_overlap"),
    n_different_chr =
      sum(coordinate_status == "different_chromosome")
  )